In [1]:
import os
import re
import shutil
from pathlib import Path
from typing import Optional, List, Dict, Any

import pandas as pd

In [2]:
os.chdir('minfin SEBRA files')

In [2]:
CSV_COLUMNS = [
    "Ref",
    "Operations Code",
    "Operations Description",
    "Operations Amount (EUR)",
    "Organization Name",
    "Start Date",
    "End Date",
    "Organization ID",
]


def _is_org_header(s: Any) -> bool:
    """Organization header lines look like: '<name> ( 055******* )'."""
    if not isinstance(s, str):
        return False
    t = s.strip()
    if not t:
        return False
    # Exclude the very first sheet title row "ОБЩО ПЛАЩАНИЯ ЗА ДЕНЯ (в евро)"
    if "ОБЩО ПЛАЩАНИЯ ЗА ДЕНЯ" in t:
        return False
    return "(" in t and ")" in t


def _parse_org_name_and_id(org_line: str) -> (str, str):
    # Example: 'Национален ... ( 055******* )'
    name = org_line.split("(", 1)[0].strip()
    m = re.search(r"\((.*?)\)", org_line)
    org_id = m.group(1).strip() if m else ""
    return name, org_id


def _parse_period(period_cell: Any) -> (str, str):
    """
    Period cell looks like: 'Период: 05.01.2026 - 05.01.2026'
    Returns dates as 'dd.mm.yyyy' strings.
    """
    if not isinstance(period_cell, str):
        return "", ""
    m = re.search(r"(\d{2}\.\d{2}\.\d{4})\s*-\s*(\d{2}\.\d{2}\.\d{4})", period_cell)
    if not m:
        return "", ""
    return m.group(1), m.group(2)


def _is_code_row(val: Any) -> bool:
    """Codes look like '01 xxxx', '40 xxxx', etc."""
    if not isinstance(val, str):
        return False
    return re.match(r"^\s*\d{2}\s*xxxx\s*$", val.strip(), flags=re.IGNORECASE) is not None


def parse_sebra_xlsx(xlsx_path: str) -> pd.DataFrame:
    """
    Parse one daily SEBRA XLSX into a normalized dataframe matching the target CSV structure,
    except for 'Ref' which is assigned when writing/appending to the combined CSV.
    """
    df = pd.read_excel(xlsx_path, sheet_name=0)  # first sheet is the dated one
    if df.shape[1] < 4:
        raise ValueError(f"Unexpected format (need 4 columns): {xlsx_path}")

    c0 = df.iloc[:, 0]
    c1 = df.iloc[:, 1]
    c2 = df.iloc[:, 2]
    c3 = df.iloc[:, 3]

    rows: List[Dict[str, Any]] = []
    i = 0
    n = len(df)

    while i < n:
        cell0 = c0.iat[i]

        if _is_org_header(cell0):
            org_name, org_id = _parse_org_name_and_id(str(cell0))
            start_date, end_date = _parse_period(c2.iat[i])

            # Sometimes period might not be on the same line; look ahead a bit
            if not start_date:
                for j in range(i, min(i + 4, n)):
                    sd, ed = _parse_period(c2.iat[j])
                    if sd:
                        start_date, end_date = sd, ed
                        break

            # Advance to the table rows (skip "Код" header line etc.)
            i += 1
            # Find the "Код" header line, then start after it
            while i < n and not (isinstance(c0.iat[i], str) and str(c0.iat[i]).strip() == "Код"):
                # If we hit another org header unexpectedly, break out
                if _is_org_header(c0.iat[i]):
                    break
                i += 1
            if i < n and isinstance(c0.iat[i], str) and str(c0.iat[i]).strip() == "Код":
                i += 1  # first data row after header

            # Read code lines until "Общо" or next org header
            while i < n:
                v0 = c0.iat[i]

                if _is_org_header(v0):
                    # new org starts
                    i -= 1  # step back so outer loop sees it next
                    break

                if isinstance(v0, str) and v0.strip().startswith("Общо"):
                    break

                if _is_code_row(v0):
                    code = str(v0).strip()
                    desc = c1.iat[i]
                    amt = c3.iat[i]

                    # normalize description
                    desc_str = "" if pd.isna(desc) else str(desc).strip()

                    # normalize amount -> float
                    amt_num = pd.to_numeric(amt, errors="coerce")

                    # Only keep rows with a numeric amount (including 0)
                    if pd.notna(amt_num):
                        rows.append(
                            {
                                "Operations Code": code,
                                "Operations Description": desc_str,
                                "Operations Amount (EUR)": float(amt_num),
                                "Organization Name": org_name,
                                "Start Date": start_date,
                                "End Date": end_date,
                                "Organization ID": org_id,
                            }
                        )

                i += 1

        i += 1

    out = pd.DataFrame(rows, columns=[c for c in CSV_COLUMNS if c != "Ref"])
    return out


def process_sebra_folder(
    folder: str,
    output_csv: str,
    processed_subfolder: str = "processed",
    encoding: str = "utf-8-sig",
) -> str:
    """
    Process all SEBRA-*.xlsx files in 'folder', append to 'output_csv',
    and move successfully processed files to '<folder>/<processed_subfolder>/'.

    Returns: output_csv path
    """
    folder_path = Path(folder)
    processed_path = folder_path / processed_subfolder
    processed_path.mkdir(parents=True, exist_ok=True)

    # Determine starting Ref (continue if CSV exists)
    out_path = Path(output_csv)
    if out_path.exists():
        existing = pd.read_csv(out_path, encoding=encoding)
        max_ref = pd.to_numeric(existing["Ref"], errors="coerce").max()
        next_ref = int(max_ref) + 1 if pd.notna(max_ref) else 1
    else:
        next_ref = 1

    # Find files like SEBRA-05012026.xlsx
    xlsx_files = sorted(folder_path.glob("SEBRA-*.xlsx"))

    appended_any = False

    for xlsx in xlsx_files:
        # Skip anything already inside processed folder
        if processed_path in xlsx.parents:
            continue

        try:
            daily = parse_sebra_xlsx(str(xlsx))
            if daily.empty:
                # Still consider it "processed" if format was okay but nothing to add
                shutil.move(str(xlsx), str(processed_path / xlsx.name))
                continue

            # Assign Ref
            daily.insert(0, "Ref", range(next_ref, next_ref + len(daily)))
            next_ref += len(daily)

            # Append / write
            if out_path.exists():
                daily.to_csv(out_path, mode="a", header=False, index=False, encoding=encoding)
            else:
                # Ensure column order exactly matches target
                daily = daily[CSV_COLUMNS]
                daily.to_csv(out_path, mode="w", header=True, index=False, encoding=encoding)

            appended_any = True

            # Move to processed after successful append
            shutil.move(str(xlsx), str(processed_path / xlsx.name))

        except Exception as e:
            # Leave the file in place so you can inspect/retry; raise for visibility
            raise RuntimeError(f"Failed processing {xlsx.name}: {e}") from e

    # If CSV existed but we only appended, ensure columns match (optional safety)
    if out_path.exists() and appended_any:
        # no-op; kept for clarity
        pass

        # Append Category column from categories.csv
    categories_path = folder_path / "categories.csv"

    if categories_path.exists() and out_path.exists():
        output_df = pd.read_csv(out_path, encoding=encoding)
        categories_df = pd.read_csv(categories_path, encoding=encoding)

        # Keep only needed columns
        categories_df = categories_df[["Organization ID", "Category"]]

        # Clean join keys
        output_df["Organization ID"] = output_df["Organization ID"].astype(str).str.strip()
        categories_df["Organization ID"] = categories_df["Organization ID"].astype(str).str.strip()

        if "Category" in output_df.columns:
            output_df = output_df.drop(columns=["Category"])

        # Merge category
        output_df = output_df.merge(
            categories_df,
            on="Organization ID",
            how="left"
        )

        # Save updated file
        output_df.to_csv(out_path, index=False, encoding=encoding)

    return str(out_path)

In [4]:
parse_sebra_xlsx('./Data/SEBRA-09042026.xlsx')

# output = process_sebra_folder(
#     folder="minfin SEBRA files",
#     output_csv="SEBRA2026EUR.csv",
#     encoding="utf-8-sig",
# )

,Operations Code,Operations Description,Operations Amount (EUR),Organization Name,Start Date,End Date,Organization ID
0,01 xxxx,"Заплати, възнаграждения и други плащания за пе...",17534.74,Народно събрание,09.04.2026,09.04.2026,001*******
1,10 xxxx,Издръжка,492960.94,Народно събрание,09.04.2026,09.04.2026,001*******
2,90 xxxx,Възстановени приходи,1436.55,Народно събрание,09.04.2026,09.04.2026,001*******
3,01 xxxx,"Заплати, възнаграждения и други плащания за пе...",55594.53,Министерски съвет,09.04.2026,09.04.2026,003*******
4,10 xxxx,Издръжка,83802.68,Министерски съвет,09.04.2026,09.04.2026,003*******
...,...,...,...,...,...,...,...
217,89 xxxx,Друго финансиране,2535183.76,Национален фонд - Механизъм за възстановяване ...,09.04.2026,09.04.2026,983*******
218,10 xxxx,Издръжка,3095.90,Национален фонд - Средства от Европейския съюз,09.04.2026,09.04.2026,987*******
219,30 xxxx,Текущи субсидии за предприятия,704998.16,Национален фонд - Средства от Европейския съюз,09.04.2026,09.04.2026,987*******
220,60 xxxx,Трансфери за бюджетни и извънбюджетни сметки,1971681.79,Национален фонд - Средства от Европейския съюз,09.04.2026,09.04.2026,987*******
